In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [7]:
import sys
from pathlib import Path

# Make sure repo root is on the path
repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import asyncio
import json
import logging
import os

import dotenv
import httpx
import pandas as pd
from datasets import load_dataset
from IPython.display import display

from src.data_generation.generate_attacks import generate_all

dotenv.load_dotenv(repo_root / ".env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s — %(message)s", datefmt="%H:%M:%S")
print("Imports OK")

Imports OK


## 1. Environment checks

In [ ]:
# Check OpenAI key
openai_key = os.environ.get("OPENAI_API_KEY", "")
assert openai_key, "OPENAI_API_KEY not set — fill in .env"
print(f"OpenAI key: {'*' * (len(openai_key) - 4)}{openai_key[-4:]}")

# Check vLLM server
vllm_url = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")
try:
    r = httpx.get(f"{vllm_url}/models", headers={"Authorization": "Bearer token-abc123"}, timeout=5)
    models = r.json()["data"]
    print(f"vLLM server OK — serving: {[m['id'] for m in models]}")
except Exception as e:
    print(f"vLLM server NOT reachable at {vllm_url}: {e}")
    print("Run: bash scripts/launch_vllm.sh")

## 2. Load JBB-Behaviors

In [ ]:
ds = load_dataset("JailbreakBench/JBB-Behaviors", name="behaviors", split="harmful")

objectives = [
    {
        "objective": row["Goal"],
        "pair_id": row["Index"],
        "category": row["Category"],
        "behavior_id": row["Behavior"],
    }
    for row in ds
]

df = pd.DataFrame(objectives)
print(f"{len(objectives)} objectives loaded")
display(df[["pair_id", "category", "behavior_id", "objective"]].head(10))

In [ ]:
# Category breakdown
display(df["category"].value_counts().rename("count").to_frame())

## 3. Configuration

Edit the cell below before running.

In [ ]:
CONFIG = dict(
    framework       = "crescendo",
    model_shortname = "llama3_8b",
    target_model_id = "meta-llama/Llama-3.1-8B-Instruct",
    n_turns         = 3,
    n_attempts      = 1,
    concurrency     = 5,       # simultaneous conversations
    attacker_model  = os.environ.get("ATTACKER_MODEL", "gpt-4o-mini"),
    judge_model     = os.environ.get("JUDGE_MODEL", "gpt-4o"),
    vllm_base_url   = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1"),
    output_dir      = repo_root / "data" / "conversations" / "harmful",
    resume          = True,    # skip already-completed conversations
    # Set max_objectives to a small number for a smoke test, None for all 100
    max_objectives  = 3,
)

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 4. Run generation

Each conversation is saved immediately to `data/conversations/harmful/` as it completes, so partial runs are safe to interrupt and resume.

In [ ]:
subset = objectives[:CONFIG["max_objectives"]] if CONFIG["max_objectives"] else objectives
print(f"Running {CONFIG['framework']} on {len(subset)} objectives × {CONFIG['n_attempts']} attempts = {len(subset) * CONFIG['n_attempts']} conversations")

conversations = await generate_all(
    objectives      = subset,
    framework       = CONFIG["framework"],
    model_shortname = CONFIG["model_shortname"],
    target_model_id = CONFIG["target_model_id"],
    n_turns         = CONFIG["n_turns"],
    n_attempts      = CONFIG["n_attempts"],
    output_dir      = CONFIG["output_dir"],
    concurrency     = CONFIG["concurrency"],
    openai_api_key  = openai_key,
    vllm_base_url   = CONFIG["vllm_base_url"],
    attacker_model  = CONFIG["attacker_model"],
    judge_model     = CONFIG["judge_model"],
    resume          = CONFIG["resume"],
)

print(f"\nDone. {len(conversations)} conversations returned.")

## 5. Results summary

In [ ]:
results_df = pd.DataFrame([
    {
        "pair_id":        c["objective_pair_id"],
        "behavior_id":    next((o["behavior_id"] for o in objectives if o["pair_id"] == c["objective_pair_id"]), ""),
        "category":       next((o["category"]    for o in objectives if o["pair_id"] == c["objective_pair_id"]), ""),
        "verdict":        c["verdict"],
        "executed_turns": c["executed_turns"],
        "objective":      c["objective"][:80],
    }
    for c in conversations
])

display(results_df)
print()
print(results_df["verdict"].value_counts().to_string())

In [ ]:
# Inspect a specific conversation
PAIR_ID = results_df["pair_id"].iloc[0]   # change as needed

conv_files = list(CONFIG["output_dir"].glob(f"*_pair{PAIR_ID:03d}_*.json"))
if conv_files:
    conv = json.loads(conv_files[0].read_text())
    print(f"Objective: {conv['objective']}")
    print(f"Verdict:   {conv['verdict']} ({conv['executed_turns']}/{conv['n_turns']} turns)\n")
    for turn in conv["turns"]:
        role = turn["role"].upper()
        print(f"[Turn {turn['turn_idx']} — {role}]")
        print(turn["content"])
        print()